In [1]:
### Imports ###

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import pandas as pd

In [2]:
df = pd.read_csv("../data/master_labels_volume.csv")
dam_names = df['dam_name'].unique()

for val_dam in dam_names:
    print(f"--- FOLD: {val_dam} ---")
    
    # Create the separate DataFrames for the current fold
    df_val = df[df['dam_name'] == val_dam]
    df_train = df[df['dam_name'] != val_dam]

    # Save them to disk
    df_train.to_csv(f"../data/loocv_labels/train_labels_{val_dam}.csv", index=False)
    df_val.to_csv(f"../data/loocv_labels/val_labels_{val_dam}.csv", index=False)

    print(f"Training images: {len(df_train)}")
    print(f"Validation (Left-Out) images: {len(df_val)}\n")

--- FOLD: baells ---
Training images: 1949
Validation (Left-Out) images: 335

--- FOLD: boadella ---
Training images: 2114
Validation (Left-Out) images: 170

--- FOLD: foix ---
Training images: 1987
Validation (Left-Out) images: 297

--- FOLD: llosa de cavall ---
Training images: 1983
Validation (Left-Out) images: 301

--- FOLD: riudecanyes ---
Training images: 1987
Validation (Left-Out) images: 297

--- FOLD: sant ponç ---
Training images: 1983
Validation (Left-Out) images: 301

--- FOLD: sau ---
Training images: 2141
Validation (Left-Out) images: 143

--- FOLD: siurana ---
Training images: 1987
Validation (Left-Out) images: 297

--- FOLD: susqueda ---
Training images: 2141
Validation (Left-Out) images: 143



In [3]:
# Device configuration

device = torch.device('cpu')
if torch.cuda.is_available(): device = torch.device('cuda')
if torch.backends.mps.is_available(): device = torch.device("mps")
print (f"Using device: {device}")

Using device: cpu


In [4]:
# Hyperparameters

BATCH_SIZE = 8
LEARNING_RATE = 0.001
EPOCHS = 1     
LOSS_MODE = 'log_mse'    # choose between mse, mape, and log_mse   
PATIENCE = 12   

In [9]:
# Define an EarlyStopping class that will stop
# the training if the validation loss doesn't 
# improve after a given patience
 
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0, verbose=True, path='best_model.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.path = path

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'    [EarlyStopping] No improvement. Patience: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        if self.verbose:
            print(f'    [EarlyStopping] Val loss decreased. Saving model to {self.path}...')
        torch.save(model.state_dict(), self.path)

In [10]:
# Define the DamDataset class, inheriting attributes
# from the Dataset class. 

class DamDataset(Dataset):
    def __init__(self, data, max_size=500):
        # Modified to accept either a CSV path or a Pandas DataFrame directly for LOOCV loops
        if isinstance(data, str):
            self.df = pd.read_csv(data)
        else:
            self.df = data.copy().reset_index(drop=True)
        self.max_size = max_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        with rasterio.open(row['filepath']) as src:
            img = src.read(1)
        
        # Replace any NaN math errors with -1.0 (Dry Land)
        img = np.nan_to_num(img, nan=-1.0)

        # Clean API artifacts and convert to Tensor (1, H, W)
        img = np.clip(img, -1.0, 1.0)
        tensor = torch.from_numpy(img).unsqueeze(0).float()
        
        # Linear scaling. We obtain values from 0 to 1
        raw_volume = row['volum']

        if(LOSS_MODE=='log_mse'):
            log_volume = np.log1p(raw_volume) 
            label = torch.tensor(log_volume, dtype=torch.float32)
        else:
            label = torch.tensor(raw_volume / 300.0, dtype=torch.float32)

        # Pad to exactly 500x500 using -1.0 (Dry Land)
        pad_h = self.max_size - tensor.shape[1]
        pad_w = self.max_size - tensor.shape[2]
        padded_tensor = F.pad(tensor, (0, pad_w, 0, pad_h), value=-1.0)
        
        return padded_tensor, label

In [11]:
# Definition of the Neural Net

class DamCNN(nn.Module):
    def __init__(self):
        super(DamCNN, self).__init__()

        # Block 1: Input is (1, 500, 500) -> Output is (16, 250, 250)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Block 2: Input is (16, 250, 250) -> Output is (32, 125, 125)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Block 3: Input is (32, 125, 125) -> Output is (64, 62, 62)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Block 4: Input is (64, 62, 62) -> Output is (128, 31, 31)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Destroy spatial memorization.
        # It takes 128 maps of 31x31 pixels and turns them into exactly 128 numbers.
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        
        self.fc1 = nn.Linear(128, 256)
        self.relu_fc = nn.ReLU()
        
        # Final layer outputs a single number (the volume)
        self.fc2 = nn.Linear(in_features=256, out_features=1)
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.pool4(self.relu4(self.conv4(x)))
        
        x = self.global_pool(x)
        x = self.flatten(x)
        
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)
        
        return x.view(-1, 1)

In [12]:
# Definition of the MAPE loss (Mean Absolute Percentage Error)
def mape_loss(predictions, targets):
    epsilon = 1e-8  # Prevents division by zero
    loss = torch.abs((targets - predictions) / (targets + epsilon))
    return torch.mean(loss)

In [14]:
# Dictionary to store the best validation loss for each fold
loocv_results = {}

for val_dam in dam_names:
    print(f"STARTING FOLD: Validating on {val_dam}")

    # 1. Datasets & Dataloaders for current fold
    train_dataset = DamDataset(data=f"../data/loocv_labels/train_labels_{val_dam}.csv", max_size=500)
    val_dataset = DamDataset(data=f"../data/loocv_labels/val_labels_{val_dam}.csv", max_size=500)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 2. DESTROY AND REBUILD THE MODEL (CRITICAL STEP)
    # Re-instantiate the model to reset weights for the new fold
    model = DamCNN().to(device) 
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # 3. Setup Early Stopping for this specific fold
    save_path = f'../models/best_model_loocv_{val_dam}_{LOSS_MODE}.pth'
    stopper = EarlyStopping(patience=PATIENCE, path=save_path)

    # 4. Training Loop
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device).view(-1, 1)
            
            optimizer.zero_grad()
            predictions = model(images)
            
            # Calculate the Error
            if(LOSS_MODE=='mape'):
                loss = mape_loss(predictions, labels)
            else:
                loss = F.mse_loss(predictions, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        avg_train_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for val_images, val_labels in val_loader:
                val_images = val_images.to(device)
                val_labels = val_labels.to(device).view(-1, 1)
                val_predictions = model(val_images)
                
                # Calculate the Error
                if(LOSS_MODE=='mape'):
                    v_loss = mape_loss(val_predictions, val_labels)
                else:
                    v_loss = F.mse_loss(val_predictions, val_labels)

                val_loss += v_loss.item()

        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {LOSS_MODE}")
        stopper(avg_val_loss, model)
        
        if stopper.early_stop:
            print(f"!!! Early stopping triggered for {val_dam}. Training halted. !!!")
            break
            
    print(f"Training finished. Best model is saved as '../models/dam_model_{LOSS_MODE}_{val_dam}.pth'")
    loocv_results[val_dam] = stopper.best_loss

for dam, score in loocv_results.items():
    print(f"{dam}: {score:.4f}")

average_error = sum(loocv_results.values()) / len(loocv_results)
print(f"\n🌍 GLOBAL MEAN VALIDATION LOSS ({LOSS_MODE}): {average_error:.4f}")

STARTING FOLD: Validating on baells


KeyboardInterrupt: 